In [4]:

# ==========================================================
# PART 4 — GENAI
# TASK 4.1 — ONE VALIDATED CALL 
# ==========================================================

# STEP 1: Install required libraries

%pip install -q google-genai pydantic pandas


# STEP 2: Import libraries

import pandas as pd
import json
import getpass

from google import genai
from google.genai import types


# STEP 3: Load follow-up notes dataset

followup_notes = pd.read_csv("C:\\Users\\Dipali\\CareLopp_Clinic_Project\\followup_notes.csv")

print("Dataset loaded successfully!")
print(followup_notes.head())


# STEP 4: Define Gemini Flash model

MODEL_NAME = "gemini-3.5-flash"


# STEP 5: Define JSON response schema

SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"}
    },
    "required": [
        "root_cause",
        "confidence",
        "summary",
        "recommended_action"
    ]
}


# STEP 6: Define system instruction

SYSTEM_TEXT = """
You are an expert healthcare appointment analyst.
Analyse patient follow-up notes carefully.
Identify the main reason why the patient missed the appointment.
Provide accurate and evidence-based information.
Do not invent information.
"""


# STEP 7: Enter Gemini API key

API_KEY = getpass.getpass("MyGeminiKey: ")

client = genai.Client(api_key=API_KEY)

print("Gemini client connected successfully!")


# STEP 8: Create analyse_note() function

def analyse_note(note_text):

    global LAST_RAW_RESPONSE

    prompt = f"""
    CONTEXT:
    Analyse healthcare appointment follow-up notes.

    ROLE:
    Act as an expert healthcare data analyst.

    ACTION:
    Identify the main reason why the patient missed
    their appointment.

    Allowed root causes:
    - forgot_or_no_reminder
    - distance_or_transport
    - cost_or_fee
    - illness_or_emergency
    - work_or_schedule
    - other

    TONE:
    Be professional, accurate and evidence-based.
    Do not invent information.

    FORMAT:
    Return exactly four JSON fields:
    root_cause: string
    confidence: number between 0 and 1
    summary: maximum 25 words
    recommended_action: string

    PATIENT NOTE:
    {note_text}
    """

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1500,
            response_mime_type="application/json",
            response_schema=SCHEMA
        )
    )

    # Store original Gemini JSON response
    LAST_RAW_RESPONSE = response.text

    # Convert JSON into dictionary
    data = json.loads(response.text)

    return data



# STEP 9: Retrieve first follow-up note N001

note_n001 = followup_notes["note_text"].iloc[0]

print("\nN001 Follow-up Note:")
print(note_n001)


# STEP 10: Analyse N001

result = analyse_note(note_n001)


# STEP 11: Print parsed dictionary

print("\nN001 Analysis:")
print(result)


Note: you may need to restart the kernel to use updated packages.
Dataset loaded successfully!
  note_id appointment_id                                          note_text
0    N001          A0872  Patient said she never got any reminder messag...
1    N002          A0897  Lives far from the clinic and could not find a...
2    N003          A0526  Patient felt the consultation fee was too high...
3    N004          A0926                                                NaN
4    N005          A0534  Had high fever and body ache on the day, staye...


AQ.Ab8RN6Kz0zZD4LzpCTzh0GeBuPu6Hd87Rd08iL2udjtd9WIKzw:  ········


Gemini client connected successfully!

N001 Follow-up Note:
Patient said she never got any reminder message and completely forgot about the visit. Wants to rebook for next week.

N001 Analysis:
{'root_cause': 'forgot_or_no_reminder', 'confidence': 0.95, 'summary': 'Patient missed the appointment because she forgot and did not receive a reminder message. She requested to reschedule for next week.', 'recommended_action': 'Reschedule the appointment for next week and verify contact details to ensure automated reminder messages are correctly delivered.'}


In [6]:
# ============================================================
# PART 4 - TASK 4.2
# VALIDATE AND PROCESS THE FIRST 10 FOLLOW-UP NOTES
# ============================================================

# STEP 1: Import libraries
import time
from pydantic import BaseModel, Field


# STEP 2: Load dataset

followup_notes = pd.read_csv("C:\\Users\\Dipali\\CareLopp_Clinic_Project\\followup_notes.csv")

print("Dataset loaded successfully!")
print(followup_notes.head())
print("Total rows:", len(followup_notes))


# STEP 3: Define allowed root causes

ALLOWED_ROOT_CAUSES = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other"
]


# STEP 4: Define Pydantic model

class FollowupInsight(BaseModel):

    root_cause: str = Field(min_length=3)

    confidence: float = Field(ge=0, le=1)

    summary: str = Field(min_length=5)

    recommended_action: str = Field(min_length=5)


# STEP 5: Create results list

results = []

api_calls = 0


# STEP 6: Process first 10 notes

for index, row in followup_notes.head(10).iterrows():

    note_id = row["note_id"]
    appointment_id = row["appointment_id"]
    note_text = row["note_text"]

    print(f"\nProcessing: {note_id}")

    # Check blank, missing or whitespace-only notes

    if pd.isna(note_text) or str(note_text).strip() == "":

        print(f"Blank note skipped: {note_id}")

        results.append({
            "note_id": note_id,
            "appointment_id": appointment_id,
            "root_cause": "other",
            "confidence": 0.0,
            "summary": "No note recorded",
            "recommended_action": "Re-call patient",
            "validation_status": "skipped_blank",
            "raw_response": ""
        })

        continue

    data = None
    raw_response = ""

    try:

        # Call Gemini API using Task 4.1 function

        api_calls += 1

        LAST_RAW_RESPONSE = ""

        data = analyse_note(str(note_text))

        # Preserve original Gemini JSON response

        raw_response = LAST_RAW_RESPONSE

        # Validate with Pydantic

        validated = FollowupInsight(**data)

        # Check allowed root cause

        if validated.root_cause not in ALLOWED_ROOT_CAUSES:

            raise ValueError(
                f"Unexpected root cause: {validated.root_cause}"
            )

        # Store valid response

        results.append({
            "note_id": note_id,
            "appointment_id": appointment_id,
            "root_cause": validated.root_cause,
            "confidence": validated.confidence,
            "summary": validated.summary,
            "recommended_action": validated.recommended_action,
            "validation_status": "valid",
            "raw_response": raw_response
        })

        print(f"Valid response: {note_id}")

        print(validated.model_dump())

    except Exception as e:

        print(f"Invalid response: {note_id}")
        print("Error:", str(e))

        # Preserve original reply and copy available fields

        if not raw_response:
            raw_response = LAST_RAW_RESPONSE

        if isinstance(data, dict):

            root_cause = data.get("root_cause", "")
            confidence = data.get("confidence", "")
            summary = data.get("summary", "")
            recommended_action = data.get(
                "recommended_action", ""
            )

        else:

            root_cause = ""
            confidence = ""
            summary = ""
            recommended_action = ""

        results.append({
            "note_id": note_id,
            "appointment_id": appointment_id,
            "root_cause": root_cause,
            "confidence": confidence,
            "summary": summary,
            "recommended_action": recommended_action,
            "validation_status": "invalid",
            "raw_response": raw_response
        })

    # Pause between API calls

    if api_calls < sum(
        not pd.isna(n) and str(n).strip() != ""
        for n in followup_notes.head(10)["note_text"]
    ):
        print("Waiting 13 seconds...")
        time.sleep(13)


# STEP 7: Create final DataFrame

columns = [
    "note_id",
    "appointment_id",
    "root_cause",
    "confidence",
    "summary",
    "recommended_action",
    "validation_status",
    "raw_response"
]

notes_analysed = pd.DataFrame(results, columns=columns)


# STEP 8: Display final analysis

print("\n========== FINAL ANALYSIS ==========")

display(notes_analysed)

print("\nNumber of rows:", len(notes_analysed))
print("Number of columns:", len(notes_analysed.columns))


# STEP 9: Validation status counts

print("\nValidation status counts:")

print(notes_analysed["validation_status"].value_counts())


# STEP 10: Check N004

print("\nN004 validation:")

display(
    notes_analysed[
        notes_analysed["note_id"] == "N004"
    ]
)


# STEP 11: Confidence validation

valid_rows = notes_analysed[
    notes_analysed["validation_status"] == "valid"
]

confidence_check = valid_rows["confidence"].between(
    0, 1
).all()

print("\nConfidence validation passed:", confidence_check)


# STEP 12: Save CSV

notes_analysed.to_csv(
    "notes_analysed.csv",
    index=False
)

print("\nnotes_analysed.csv saved successfully!")


# STEP 13: API call count

print("\nAPI calls:", api_calls)

print("\nAPI calls COMPLETED SUCCESSFULLY!")


Dataset loaded successfully!
  note_id appointment_id                                          note_text
0    N001          A0872  Patient said she never got any reminder messag...
1    N002          A0897  Lives far from the clinic and could not find a...
2    N003          A0526  Patient felt the consultation fee was too high...
3    N004          A0926                                                NaN
4    N005          A0534  Had high fever and body ache on the day, staye...
Total rows: 30

Processing: N001
Valid response: N001
{'root_cause': 'forgot_or_no_reminder', 'confidence': 0.98, 'summary': 'The patient forgot the appointment because she did not receive a reminder message and requested to rebook for next week.', 'recommended_action': "Reschedule the appointment for next week and verify the patient's contact preferences to ensure automated reminders are properly configured."}
Waiting 13 seconds...

Processing: N002
Valid response: N002
{'root_cause': 'distance_or_transport',

,note_id,appointment_id,root_cause,confidence,summary,recommended_action,validation_status,raw_response
0,N001,A0872,forgot_or_no_reminder,0.98,The patient forgot the appointment because she...,Reschedule the appointment for next week and v...,valid,"{""root_cause"":""forgot_or_no_reminder"",""confide..."
1,N002,A0897,distance_or_transport,0.98,Patient missed the appointment because they li...,Offer telehealth appointments or provide infor...,valid,"{""root_cause"":""distance_or_transport"",""confide..."
2,N003,A0526,cost_or_fee,0.98,The patient skipped the appointment due to hig...,Provide information on available financial ass...,valid,"{""root_cause"":""cost_or_fee"",""confidence"":0.98,..."
3,N004,A0926,other,0.0,No note recorded,Re-call patient,skipped_blank,
4,N005,A0534,illness_or_emergency,0.98,Patient missed the appointment due to high fev...,Follow up with the patient in 2-3 days to chec...,valid,"{""root_cause"":""illness_or_emergency"",""confiden..."
5,N006,A0507,work_or_schedule,0.98,Patient missed the appointment due to an urgen...,Reschedule the appointment for an evening time...,valid,"{""root_cause"":""work_or_schedule"",""confidence"":..."
6,N007,A0333,forgot_or_no_reminder,0.95,Patient forgot the appointment due to not rece...,Reschedule the appointment and verify the phon...,valid,"{""root_cause"":""forgot_or_no_reminder"",""confide..."
7,N008,A0222,distance_or_transport,0.95,Patient missed the appointment due to a city b...,Provide transport assistance options or evalua...,valid,"{""root_cause"":""distance_or_transport"",""confide..."
8,N009,A0977,other,0.85,Patient was unreachable after three call attem...,Attempt contact via alternative channels such ...,valid,"{""root_cause"":""other"",""confidence"":0.85,""summa..."
9,N010,A0596,,,,,invalid,



Number of rows: 10
Number of columns: 8

Validation status counts:
validation_status
valid            8
skipped_blank    1
invalid          1
Name: count, dtype: int64

N004 validation:


,note_id,appointment_id,root_cause,confidence,summary,recommended_action,validation_status,raw_response
3,N004,A0926,other,0.0,No note recorded,Re-call patient,skipped_blank,



Confidence validation passed: True

notes_analysed.csv saved successfully!

API calls: 9

API calls COMPLETED SUCCESSFULLY!


In [11]:

# ============================================================
# TASK 4.3 — BREAK IT ON PURPOSE
# ============================================================

import json
import pandas as pd
from pydantic import ValidationError
from IPython.display import display


# ------------------------------------------------------------
# PART A — TEST PYDANTIC WITH TWO DELIBERATELY BAD REPLIES
# ------------------------------------------------------------

print("=" * 60)
print("PART A — PYDANTIC VALIDATION TESTING")
print("=" * 60)

# Bad reply 1: confidence is greater than 1
bad_reply_1 = {
    "root_cause": "forgot_or_no_reminder",
    "confidence": 1.7,
    "summary": "Patient forgot the appointment.",
    "recommended_action": "Send a reminder."
}

# Bad reply 2: recommended_action is missing
bad_reply_2 = {
    "root_cause": "distance_or_transport",
    "confidence": 0.85,
    "summary": "Patient lives far away."
}

bad_replies = [bad_reply_1, bad_reply_2]

for i, reply in enumerate(bad_replies, start=1):
    print(f"\nTesting bad reply {i}:")

    try:
        validated = FollowupInsight(**reply)
        print("Unexpectedly valid:", validated)

    except ValidationError as e:
        print("ValidationError caught:")
        print(e)


# ------------------------------------------------------------
# PART B — COMPARE V1 AND V2 PROMPTS
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("PART B — PROMPT COMPARISON")
print("=" * 60)

# Allowed root-cause categories
allowed_root_causes = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other"
]

# Retrieve the note text for N007 and N009
notes_to_test = followup_notes[
    followup_notes["note_id"].isin(["N007", "N009"])
]

if set(notes_to_test["note_id"]) != {"N007", "N009"}:
    raise ValueError("N007 or N009 was not found in followup_notes.")

# V1: Simple prompt
def create_v1_prompt(note_text):
    return f"""
Classify this note:

{note_text}
"""


# V2: CRAFT prompt with allowed categories and instruction not to guess
def create_v2_prompt(note_text):
    return f"""
CONTEXT:
We are analysing patient appointment no-show notes.

ROLE:
You are an experienced healthcare data analyst.

ACTION:
Read the patient note and identify the main reason for missing
the appointment.

Use only one of these root_cause values:
{allowed_root_causes}

If the note does not provide enough information to identify
a reliable reason, choose "other". Do not guess.

TONE:
Be clear, factual and concise.

FORMAT:
Return only valid JSON with exactly these four fields:
- root_cause: string
- confidence: number between 0 and 1
- summary: string of maximum 25 words
- recommended_action: string

PATIENT NOTE:
{note_text}
"""


# Function to call Gemini with the same schema and settings
# for both prompts. Only the prompt text changes.

def test_prompt(note_text, prompt_version, prompt):

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0,
            max_output_tokens=1500,
            response_mime_type="application/json",
            response_schema=SCHEMA
        )
    )

    raw_response = response.text

    try:
        result = json.loads(raw_response)

        root_cause = result.get("root_cause", "")

        # Check whether root_cause belongs to allowed categories
        if root_cause in allowed_root_causes:
            validation_status = "valid"
        else:
            validation_status = "invalid"

        return {
            "note_id": prompt_version["note_id"],
            "prompt": prompt_version["prompt"],
            "root_cause": root_cause,
            "confidence": result.get("confidence"),
            "validation_status": validation_status,
            "raw_response": raw_response
        }

    except (json.JSONDecodeError, AttributeError, TypeError) as e:

        return {
            "note_id": prompt_version["note_id"],
            "prompt": prompt_version["prompt"],
            "root_cause": "",
            "confidence": None,
            "validation_status": "invalid",
            "raw_response": raw_response
        }


# Run both prompts for N007 and N009
comparison_results = []

for _, row in notes_to_test.iterrows():

    note_id = row["note_id"]
    note_text = row["note_text"]

    print(f"\nProcessing {note_id}...")

    # V1 — Simple prompt
    v1_prompt = create_v1_prompt(note_text)

    v1_result = test_prompt(
        note_text,
        {"note_id": note_id, "prompt": "V1"},
        v1_prompt
    )

    comparison_results.append(v1_result)

    print("V1 result:", v1_result["root_cause"],
          "| Status:", v1_result["validation_status"])

    # V2 — CRAFT prompt
    v2_prompt = create_v2_prompt(note_text)

    v2_result = test_prompt(
        note_text,
        {"note_id": note_id, "prompt": "V2"},
        v2_prompt
    )

    comparison_results.append(v2_result)

    print("V2 result:", v2_result["root_cause"],
          "| Status:", v2_result["validation_status"])


# Create a small comparison table
comparison_df = pd.DataFrame(comparison_results)

comparison_table = comparison_df[
    ["note_id", "prompt", "root_cause",
     "confidence", "validation_status"]
]

print("\nFOUR PROMPT COMPARISON RESULTS:")
display(comparison_table)


# ------------------------------------------------------------
# PART C — TOTAL API CALLS ACROSS TASKS 4.1 TO 4.3
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("PART C — TOTAL API CALL COUNT")
print("=" * 60)

# Task 4.1: One initial test call
task_4_1_calls = 1

# Task 4.2: Nine nonblank notes in the first 10 rows.
# This assumes Task 4.2 was completed once.
task_4_2_calls = 9

# Task 4.3: Four calls (N007 V1, N007 V2, N009 V1, N009 V2)
task_4_3_calls = 4

total_api_calls = (
    task_4_1_calls +
    task_4_2_calls +
    task_4_3_calls
)

print("Task 4.1 API calls:", task_4_1_calls)
print("Task 4.2 API calls:", task_4_2_calls)
print("Task 4.3 API calls:", task_4_3_calls)
print("-" * 40)
print("TOTAL API CALLS:", total_api_calls)


# ------------------------------------------------------------
# SAVE TASK 4.3 COMPARISON RESULTS
# ------------------------------------------------------------

comparison_df.to_csv(
    "task_4_3_prompt_comparison.csv",
    index=False
)

print("\nTask 4.3 comparison saved successfully!")


PART A — PYDANTIC VALIDATION TESTING

Testing bad reply 1:
ValidationError caught:
1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal

Testing bad reply 2:
ValidationError caught:
1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'distance_...atient lives far away.'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing

PART B — PROMPT COMPARISON

Processing N007...


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://aistudio.google.com/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash, description: Generate content requests per day limit exceeded on Free Tier. See https://aistudio.google.com/docs/rate-limits.\nPlease retry in 3h29m53.805027969s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://aistudio.google.com/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 'Generate content requests per day limit exceeded on Free Tier. See https://aistudio.g

In [10]:
# ============================================================
# TASK 4.4 — CORRECT N001 AND N007 CLASSIFICATION
# ============================================================

import pandas as pd
import json
import time

from pydantic import BaseModel, Field, ValidationError
from google.genai import types


# ============================================================
# STEP 1: LOAD YOUR EXISTING NOTES FILE
# ============================================================

notes = pd.read_csv("notes_analysed.csv")

print("Original notes dataset:")
display(notes.head(10))


# ============================================================
# STEP 2: DEFINE ALLOWED ROOT CAUSES
# ============================================================

allowed_root_causes = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other"
]


# ============================================================
# STEP 3: DEFINE PYDANTIC VALIDATION MODEL
# ============================================================

class FollowupInsight(BaseModel):

    root_cause: str = Field(min_length=3)

    confidence: float = Field(ge=0, le=1)

    summary: str = Field(min_length=5)

    recommended_action: str = Field(min_length=5)


# ============================================================
# STEP 4: CREATE A STRONGER CRAFT PROMPT
# ============================================================

def create_corrected_prompt(note_text):

    prompt = f"""
CONTEXT:
You are analysing healthcare appointment call-centre notes
to understand why patients missed their appointments.

ROLE:
Act as an experienced healthcare data analyst.

ACTION:
Read the patient note carefully and identify the main reason
for missing the appointment.

Choose exactly ONE root_cause from this list:

1. forgot_or_no_reminder
2. distance_or_transport
3. cost_or_fee
4. illness_or_emergency
5. work_or_schedule
6. other

CLASSIFICATION RULES:

- If the patient forgot the appointment or did not receive
  a reminder, select forgot_or_no_reminder.

- If the patient had transport or distance problems,
  select distance_or_transport.

- If the patient could not afford the appointment,
  select cost_or_fee.

- If illness or an emergency prevented attendance,
  select illness_or_emergency.

- If work or scheduling conflicts prevented attendance,
  select work_or_schedule.

- If the note is unclear, incomplete, ambiguous or does not
  provide enough evidence, select other.

- Do not guess.
- Do not invent information.
- Do not create new root-cause categories.
- Return the exact category spelling from the allowed list.

TONE:
Be factual, concise and evidence-based.

FORMAT:
Return valid JSON with exactly these four fields:
root_cause, confidence, summary, recommended_action.

The summary must contain no more than 25 words.

PATIENT NOTE:
{note_text}
"""

    return prompt


# ============================================================
# STEP 5: CREATE FUNCTION TO CALL GEMINI
# ============================================================

def analyse_note_corrected(note_text):

    prompt = create_corrected_prompt(note_text)

    response = client.models.generate_content(

        model=MODEL_NAME,

        contents=prompt,

        config=types.GenerateContentConfig(

            system_instruction=SYSTEM_TEXT,

            temperature=0,

            response_mime_type="application/json",

            response_schema=SCHEMA,

            max_output_tokens=1000

        )
    )

    raw_response = response.text

    data = json.loads(raw_response)

    return data, raw_response


# ============================================================
# STEP 6: PROCESS ONLY N001 AND N007
# ============================================================

target_notes = ["N001", "N007"]

for note_id in target_notes:

    print("\nProcessing:", note_id)

    # Find the note
    row_index = notes.index[
        notes["note_id"] == note_id
    ]

    if len(row_index) == 0:

        print("ERROR: Note not found:", note_id)

        continue

    index = row_index[0]

    # Get original note text
    appointment_id = notes.loc[index, "appointment_id"]

    # Retrieve the original note from followup_notes
    original_note = followup_notes.loc[
        followup_notes["appointment_id"] == appointment_id,
        "note_text"
    ].iloc[0]

    print("Original note:")
    print(original_note)

    try:

        # Call Gemini
        data, raw_response = analyse_note_corrected(
            str(original_note)
        )

        # Validate using Pydantic
        insight = FollowupInsight(**data)

        # Check allowed root cause
        if insight.root_cause not in allowed_root_causes:

            raise ValueError(
                "Root cause is not in the allowed list."
            )

        # Update the corresponding result using the actual response
        notes.loc[index, "root_cause"] = insight.root_cause

        notes.loc[index, "confidence"] = insight.confidence

        notes.loc[index, "summary"] = insight.summary

        notes.loc[index, "recommended_action"] = (
            insight.recommended_action
        )

        notes.loc[index, "validation_status"] = "valid"

        notes.loc[index, "raw_response"] = raw_response

        print("\nValidated result:")
        print(insight.model_dump())

    except (ValidationError, ValueError, json.JSONDecodeError) as e:

        notes.loc[index, "validation_status"] = "invalid"

        # Preserve the actual raw response if one was received.
        if "raw_response" in locals():
            notes.loc[index, "raw_response"] = raw_response

        print("Validation failed:", e)

    except Exception as e:

        print("API or processing error:", e)

    # Pause between API requests
    time.sleep(13)


# ============================================================
# STEP 7: SAVE UPDATED NOTES FILE
# ============================================================

notes.to_csv(
    "notes_analysed.csv",
    index=False
)

print("\nUpdated notes_analysed.csv saved.")


# ============================================================
# STEP 8: VERIFY N001 AND N007
# ============================================================

print("\nUpdated N001 and N007 results:")

display(
    notes.loc[
        notes["note_id"].isin(["N001", "N007"]),
        [
            "note_id",
            "appointment_id",
            "root_cause",
            "confidence",
            "validation_status"
        ]
    ]
)


# ============================================================
# STEP 9: RELOAD APPOINTMENTS DATA
# ============================================================

appointments = pd.read_csv("appointments_clean.csv")


# ============================================================
# STEP 10: MERGE BOTH DATASETS
# ============================================================

merged = pd.merge(

    notes,

    appointments,

    on="appointment_id",

    how="inner"
)

print("\nMerged dataset:")
display(merged)

print("Merged rows:", len(merged))


# ============================================================
# STEP 11: CHECK ROOT CAUSE AGAINST REMINDER STATUS
# ============================================================

root_cause_reminder = pd.crosstab(

    merged["root_cause"],

    merged["reminder_sent"],

    margins=True
)

print("\nRoot cause vs reminder status:")

display(root_cause_reminder)


# ============================================================
# STEP 12: CHECK NO REMINDER COUNT
# ============================================================

reminder_status = (
    merged["reminder_sent"]
    .astype(str)
    .str.strip()
    .str.lower()
)

no_reminder_count = (
    reminder_status == "no"
).sum()

print("\nNo reminder count:", no_reminder_count)

assert len(merged) == 10, (
    "Merged dataset must contain 10 rows."
)

assert no_reminder_count == 5, (
    "Expected 5 appointments with reminder_sent = No."
)


# ============================================================
# STEP 13: CHECK N001 AND N007
# ============================================================

forgot_notes = merged[
    merged["note_id"].isin(["N001", "N007"])
].copy()

print("\nN001 and N007 verification:")

display(
    forgot_notes[
        [
            "note_id",
            "appointment_id",
            "root_cause",
            "reminder_sent"
        ]
    ]
)

assert len(forgot_notes) == 2

assert (
    forgot_notes["root_cause"]
    == "forgot_or_no_reminder"
).all(), (
    "N001 or N007 still has an unexpected root cause."
)

forgot_reminders = (
    forgot_notes["reminder_sent"]
    .astype(str)
    .str.strip()
    .str.lower()
)

assert (forgot_reminders == "no").all(), (
    "N001 or N007 does not have reminder_sent = No."
)

print("\nPASS: N001 and N007 both have the expected classification.")

print("PASS: Both appointments have reminder_sent = No.")


# ============================================================
# STEP 14: SAVE FINAL MERGED FILES
# ============================================================

merged.to_csv(
    "merged_notes_appointments.csv",
    index=False
)

root_cause_reminder.to_csv(
    "root_cause_reminder_crosstab.csv"
)

print("\nFinal files saved successfully.")

print("1. notes_analysed.csv")
print("2. merged_notes_appointments.csv")
print("3. root_cause_reminder_crosstab.csv")

Original notes dataset:


,note_id,appointment_id,root_cause,confidence,summary,recommended_action,validation_status,raw_response
0,N001,A0872,forgot_or_no_reminder,0.98,The patient forgot the appointment because she...,Reschedule the appointment for next week and v...,valid,"{""root_cause"":""forgot_or_no_reminder"",""confide..."
1,N002,A0897,distance_or_transport,0.98,Patient missed the appointment because they li...,Offer telehealth appointments or provide infor...,valid,"{""root_cause"":""distance_or_transport"",""confide..."
2,N003,A0526,cost_or_fee,0.98,The patient skipped the appointment due to hig...,Provide information on available financial ass...,valid,"{""root_cause"":""cost_or_fee"",""confidence"":0.98,..."
3,N004,A0926,other,0.00,No note recorded,Re-call patient,skipped_blank,NaN
4,N005,A0534,illness_or_emergency,0.98,Patient missed the appointment due to high fev...,Follow up with the patient in 2-3 days to chec...,valid,"{""root_cause"":""illness_or_emergency"",""confiden..."
5,N006,A0507,work_or_schedule,0.98,Patient missed the appointment due to an urgen...,Reschedule the appointment for an evening time...,valid,"{""root_cause"":""work_or_schedule"",""confidence"":..."
6,N007,A0333,forgot_or_no_reminder,0.95,Patient forgot the appointment due to not rece...,Reschedule the appointment and verify the phon...,valid,"{""root_cause"":""forgot_or_no_reminder"",""confide..."
7,N008,A0222,distance_or_transport,0.95,Patient missed the appointment due to a city b...,Provide transport assistance options or evalua...,valid,"{""root_cause"":""distance_or_transport"",""confide..."
8,N009,A0977,other,0.85,Patient was unreachable after three call attem...,Attempt contact via alternative channels such ...,valid,"{""root_cause"":""other"",""confidence"":0.85,""summa..."
9,N010,A0596,NaN,NaN,NaN,NaN,invalid,NaN



Processing: N001
Original note:
Patient said she never got any reminder message and completely forgot about the visit. Wants to rebook for next week.
API or processing error: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://aistudio.google.com/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash, description: Generate content requests per day limit exceeded on Free Tier. See https://aistudio.google.com/docs/rate-limits.\nPlease retry in 3h31m25.1007526s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://aistudio.google.com/docs/rate-limits'}]}, {'@type': 'ty

,note_id,appointment_id,root_cause,confidence,validation_status
0,N001,A0872,forgot_or_no_reminder,0.98,valid
6,N007,A0333,forgot_or_no_reminder,0.98,valid



Merged dataset:


,note_id,appointment_id,root_cause,confidence,summary,recommended_action,validation_status,raw_response,patient_id,clinic_id,...,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status,lead_days,no_show
0,N001,A0872,forgot_or_no_reminder,0.98,The patient forgot the appointment because she...,Reschedule the appointment for next week and v...,valid,"{""root_cause"":""forgot_or_no_reminder"",""confide...",P144,C05,...,F,17.6,750,App,Consultation,No,1,No-Show,14,1
1,N002,A0897,distance_or_transport,0.98,Patient missed the appointment because they li...,Offer telehealth appointments or provide infor...,valid,"{""root_cause"":""distance_or_transport"",""confide...",P268,C07,...,M,17.2,1219,Phone,Follow-up,No,0,No-Show,5,1
2,N003,A0526,cost_or_fee,0.98,The patient skipped the appointment due to hig...,Provide information on available financial ass...,valid,"{""root_cause"":""cost_or_fee"",""confidence"":0.98,...",P006,C04,...,F,5.3,800,App,Consultation,Yes,0,No-Show,34,1
3,N004,A0926,other,0.00,No note recorded,Re-call patient,skipped_blank,NaN,P449,C04,...,M,4.5,300,Phone,Follow-up,No,0,No-Show,40,1
4,N005,A0534,illness_or_emergency,0.98,Patient missed the appointment due to high fev...,Follow up with the patient in 2-3 days to chec...,valid,"{""root_cause"":""illness_or_emergency"",""confiden...",P400,C04,...,F,4.2,550,App,Follow-up,Yes,0,No-Show,26,1
5,N006,A0507,work_or_schedule,0.98,Patient missed the appointment due to an urgen...,Reschedule the appointment for an evening time...,valid,"{""root_cause"":""work_or_schedule"",""confidence"":...",P230,C04,...,F,0.8,700,App,Consultation,Yes,2,No-Show,28,1
6,N007,A0333,forgot_or_no_reminder,0.98,Patient missed the appointment because they fo...,Reschedule the appointment and verify the pati...,valid,"{""root_cause"":""forgot_or_no_reminder"",""confide...",P472,C08,...,F,15.2,850,Walk-In,Consultation,No,2,No-Show,9,1
7,N008,A0222,distance_or_transport,0.95,Patient missed the appointment due to a city b...,Provide transport assistance options or evalua...,valid,"{""root_cause"":""distance_or_transport"",""confide...",P125,C02,...,F,15.8,650,App,Follow-up,Yes,1,No-Show,4,1
8,N009,A0977,other,0.85,Patient was unreachable after three call attem...,Attempt contact via alternative channels such ...,valid,"{""root_cause"":""other"",""confidence"":0.85,""summa...",P334,C01,...,M,3.6,800,App,Consultation,Yes,1,No-Show,4,1
9,N010,A0596,NaN,NaN,NaN,NaN,invalid,NaN,P297,C04,...,F,10.1,400,App,Follow-up,No,1,No-Show,21,1


Merged rows: 10

Root cause vs reminder status:


reminder_sent,No,Yes,All
root_cause,,,
cost_or_fee,0,1,1
distance_or_transport,1,1,2
forgot_or_no_reminder,2,0,2
illness_or_emergency,0,1,1
other,1,1,2
work_or_schedule,0,1,1
All,4,5,9



No reminder count: 5

N001 and N007 verification:


,note_id,appointment_id,root_cause,reminder_sent
0,N001,A0872,forgot_or_no_reminder,No
6,N007,A0333,forgot_or_no_reminder,No



PASS: N001 and N007 both have the expected classification.
PASS: Both appointments have reminder_sent = No.

Final files saved successfully.
1. notes_analysed.csv
2. merged_notes_appointments.csv
3. root_cause_reminder_crosstab.csv
